**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

**Day 2 — Inheritance, polymorphism, and operator overloading**

**+++++++++++++++++++++++++++++++++++++++++++++++++++++++**

**=========================================================**

**Concept 1: inheritance — reuse without repetition**

<h6><strong>Inheritance </strong>is a mechanism where a new class (child) acquires 

properties and behaviors from an existing class (parent). It enables code reuse and 

establishes a hierarchical relationship between classes.</h6>

**Key Concepts**
<h6>- Parent/Base/Superclass: The original class being inherited from

- Child/Derived/Subclass: The new class that inherits from the parent

- super(): Calls methods from the parent class

- Override: Redefining a parent method in the child class
is-a relationship: Child "is a" type of parent (e.g., CategoricalColumn is a ColumnProfile)

Here's the real-world framing: you already have a working class. 

A new thing you need to model is 90% the same, with a few differences. 

Copy-pasting the whole class and changing two methods is a maintenance disaster — fix a bug in one, forget the other.

Inheritance says: the new class is a kind of the old one. It gets everything for free and only defines what's different.<h6>


exercise c1.1

In [10]:
class ColumnProfile:
    # --- BASE CLASS ---
    def __init__(self, name: str, dtype: str, null_count: int, total_count: int):
        # Constructor - initializes the base class with 4 required attributes
        # Type hints (str, int) indicate expected data types
        self.name = name                    # Column name (e.g., "country")
        self.dtype = dtype                  # Data type (e.g., "object", "int64")
        self.null_count = null_count        # Number of missing values
        self.total_count = total_count      # Total number of rows

    @property
    def null_rate(self) -> float:
        # PROPERTY - computed attribute, accessed like an attribute (no parentheses)
        # Returns the proportion of missing values as a float
        return self.null_count / self.total_count if self.total_count else 0.0
        # If total_count is 0, return 0.0 to avoid division by zero
        # Otherwise, calculate null_rate = null_count / total_count

    @property
    def is_clean(self) -> bool:
        # PROPERTY - returns True if null_rate < 5%, False otherwise
        return self.null_rate < 0.05
        # Uses the null_rate property defined above

    def summary(self) -> str:
        # INSTANCE METHOD - returns a formatted string with column summary
        # Uses f-string with :.1% to format null_rate as percentage with 1 decimal
        return f"{self.name} [{self.dtype}] — {self.null_rate:.1%} null"
        # Example output: "country [object] — 0.2% null"


class CategoricalColumn(ColumnProfile):
    # CHILD CLASS - inherits EVERYTHING from ColumnProfile
    # The parentheses indicate inheritance from ColumnProfile
    
    def __init__(self, name, dtype, null_count, total_count, cardinality: int):
        # Child constructor - adds a new parameter 'cardinality'
        # Note: Could include type hints for parameters (name: str, etc.)
        
        super().__init__(name, dtype, null_count, total_count)
        # super() returns a reference to the parent class
        # This calls the parent's __init__ method, initializing name, dtype, null_count, total_count
        # Without this line, the parent attributes wouldn't be set!
        
        self.cardinality = cardinality
        # NEW ATTRIBUTE - only categorical columns have this
        # cardinality = number of unique values (e.g., 87 countries)

    def summary(self) -> str:
        # OVERRIDE - replaces the parent's summary method
        # Same method name, but different behavior
        
        base = super().summary()
        # Calls the parent's summary() method (the original version)
        # base = "country [object] — 0.2% null"
        
        return f"{base} | {self.cardinality} unique values"
        # Extends the parent's summary with cardinality information
        # Final output: "country [object] — 0.2% null | 87 unique values"


# CREATING AN INSTANCE OF THE CHILD CLASS
col = CategoricalColumn("country", "object", 12, 5000, 87)
# Arguments order: name, dtype, null_count, total_count, cardinality
# - "country": column name
# - "object": data type
# - 12: number of null values
# - 5000: total rows
# - 87: number of unique values

print(col.summary())
# Calls the OVERRIDDEN summary() method in CategoricalColumn
# Output: "country [object] — 0.2% null | 87 unique values"
# Note: 12/5000 = 0.0024 = 0.2%

print(col.null_rate)
# Uses INHERITED property from ColumnProfile
# Output: 0.0024 (computed as 12/5000)

print(col.is_clean)
# Uses INHERITED property from ColumnProfile
# Output: True (0.0024 < 0.05)

country [object] — 0.2% null | 87 unique values
0.0024
True



exrcise c1.2

In [7]:
class ColumnProfile:
    # --- base class from Day 1 ---
    def __init__(self, name: str, dtype: str, null_count: int, total_count: int):
        self.name = name
        self.dtype = dtype
        self.null_count = null_count
        self.total_count = total_count

    @property
    def null_rate(self) -> float:
        return self.null_count / self.total_count if self.total_count else 0.0

    @property
    def is_clean(self) -> bool:
        return self.null_rate < 0.05

    def summary(self) -> str:          # method the child will override
        return f"{self.name} [{self.dtype}] — {self.null_rate:.1%} null"


class CategoricalColumn(ColumnProfile):
    # inherits everything above; adds what's unique to categorical columns
    def __init__(self, name, dtype, null_count, total_count, cardinality: int):
        super().__init__(name, dtype, null_count, total_count)  # delegate to parent
        self.cardinality = cardinality  # new attribute only categoricals have

    def summary(self) -> str:           # OVERRIDE — same name, different behavior
        base = super().summary()        # reuse parent's string, extend it
        return f"{base} | {self.cardinality} unique values"

class NumericColumn(ColumnProfile):
    def __init__(self, name, dtype, null_count, total_count, mean: float, std: float):
        super().__init__(name, dtype, null_count, total_count)
        self.mean = mean
        self.std = std

    def summary(self):
        base = super().summary()
        return f"{base} | mean={self.mean:.2f}, std={self.std:.2f}"    


col = CategoricalColumn("country", "object", 12, 5000, 87)
row = NumericColumn('asia', 'data', 30, 3000, 20, 1.5)
print(col.summary())     # CategoricalColumn's version
print(col.null_rate)     # inherited from ColumnProfile — no code duplication
print(col.is_clean)      # also inherited
print('+'*60)
print(row.summary())
print(row.is_clean)
print(row.null_rate)

country [object] — 0.2% null | 87 unique values
0.0024
True
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
asia [data] — 1.0% null | mean=20.00, std=1.50
True
0.01


**Key Takeaways**
<h6>- Inheritance promotes code reuse - Child classes get all  parent functionality

- super() calls parent methods - Essential for extending functionality

- Override methods to specialize behavior while maintaining the same interface

- New attributes can be added in child classes

- (is-a) relationship - Child "is a" type of parent

- Single inheritance in Python (one parent class)

- Method Resolution Order (MRO) determines which method gets called

**Common Patterns:**
- Extend: Add new functionality while keeping parent behavior

- Override: Completely replace parent behavior

- Extend and Override: Use super() to get parent behavior plus add 
</h6>

**==================================================**

**Concept 2: Polymorphism — One Interface, Many Behaviors**

<h6>Polymorphism (from Greek: "many forms") is the ability of different objects 

to respond to the same method call in their own way. It allows code to work with 

objects of different types through a common interface, without knowing their specific type.</h6>

**Key Concepts**
<h6>- Same method name, different implementations

- Code that works on the parent type works on all children

- New subclasses can be added without changing existing code

- Duck typing: "If it walks like a duck and quacks like a duck, it's a duck" </h6>

example c2.1

In [8]:
# Three different column types — all subclasses of ColumnProfile
columns = [
    ColumnProfile("id", "int64", 0, 5000),
    # Creates a base ColumnProfile instance
    # Attributes: name="id", dtype="int64", null_count=0, total_count=5000
    
    CategoricalColumn("country", "object", 12, 5000, 87),
    # Creates a CategoricalColumn instance (child of ColumnProfile)
    # Inherits all parent attributes + adds cardinality=87
    
    NumericColumn("revenue", "float64", 3, 5000, 1420.5, 340.2),
    # Creates a NumericColumn instance (another child of ColumnProfile)
    # Inherits parent attributes + adds mean=1420.5, std=340.2
]

# This loop doesn't know or care what type each column is.
# It just calls .summary() — each object responds in its own way.
for col in columns:
    # POLYMORPHISM IN ACTION:
    # col could be ColumnProfile, CategoricalColumn, or NumericColumn
    # Python doesn't care — it just calls .summary() on whatever col is
    # Each class has its own implementation of summary()
    print(col.summary())   # different output per type — same call

# Output:
# id [int64] — 0.00% null
# country [object] — 0.24% null | 87 unique values
# revenue [float64] — 0.06% null | mean=1420.50, std=340.20

id [int64] — 0.0% null
country [object] — 0.2% null | 87 unique values
revenue [float64] — 0.1% null | mean=1420.50, std=340.20


**The power here:**

<h6>when you add a (DatetimeColumn) next month, the loop above needs zero changes. 

The new class just implements (summary() )and plugs in. This is how every major 

framework is designed — LangChain's (BaseTool) works exactly this way. Every tool 

(web search, calculator, Python REPL) is a subclass with its own (_run()) method. The agent calls

(tool.run(input) )without knowing which tool it is.

**What Makes This Powerful:**
- No type checking needed — just call .summary()
- Extensible — add a new column type, the loop still works

- Maintainable — change one method, all types update

- Clean code — no complex if/elif chains</h6>

**=====================================================**

**Concept 3: (isinstance) and (issubclass) — type checking at runtime**

<h6>**(isinstance())** and **(issubclass())** are built-in Python functions 

that allow you to check object types and class relationships at runtime. 

They're essential for writing flexible, safe code that needs to know what type it's dealing with.</h6>

**<h5>Key Concepts</h5>**
<h6>- isinstance(object, class): Checks if an object is an instance of a specific class (or any parent class)

- issubclass(class1, class2): Checks if one class is a subclass of another

- Runtime type checking: Allows conditional behavior based on object type

- Safe operations: Prevents errors when working with different types</h6>

example c3.1

In [9]:
# Create an instance of CategoricalColumn
col = CategoricalColumn("country", "object", 12, 5000, 87)
# col is an object with:
# - name: "country"
# - dtype: "object" 
# - null_count: 12
# - total_count: 5000
# - cardinality: 87

# isinstance checks the object's actual type — including parent types
isinstance(col, CategoricalColumn)  # True
# Checks if col is an instance of CategoricalColumn
# Returns True because col was created from CategoricalColumn

isinstance(col, ColumnProfile)      # True — it IS a ColumnProfile too
# Checks if col is an instance of ColumnProfile (the parent class)
# Returns True because CategoricalColumn inherits from ColumnProfile
# This is the "is-a" relationship: a CategoricalColumn IS A ColumnProfile

isinstance(col, NumericColumn)      # False
# Checks if col is an instance of NumericColumn
# Returns False because col is a categorical column, not numeric

# issubclass checks the class hierarchy, not an object
issubclass(CategoricalColumn, ColumnProfile)  # True
# Checks if CategoricalColumn is a subclass of ColumnProfile
# Returns True because CategoricalColumn inherits from ColumnProfile

issubclass(ColumnProfile, CategoricalColumn)  # False — parent is not child
# Checks if ColumnProfile is a subclass of CategoricalColumn
# Returns False because the parent class cannot be a child of its child
# Inheritance only goes one way: child -> parent

False

exercise c3.2

In [19]:
def profile_column(columns: list) -> dict:
    result = {
        'base': [],       
        'category': [],
        'numerical': []
    }

    for col in columns:
        if isinstance(col, ColumnProfile):
            result['base'].append(col.name)
        elif isinstance(col, CategoricalColumn):
            result['category'].append(col.name)
        elif isinstance(col, NumericColumn):
            result['numerical'].append(col.name)

    return result 

print(profile_column(columns))


{'base': ['id', 'country', 'revenue'], 'category': [], 'numerical': []}


**=====================================================**

**Concept 4: Dunder Methods — Making Objects Behave Like Built-ins**

<h6>Day 1 covered __repr__, __str__, __eq__. Today: the ones that make your objects behave like built-in Python types.

__len__, __contains__, __iter__ — make your object feel like a collection:

Dunder methods (double underscore methods, also called "magic methods") 

are special methods Python calls automatically when you use built-in syntax like len(), in, for, indexing [], etc.

They let your custom objects behave like native Python types.</h6>

**<h5>Key Concepts </h5>**
<h6>Dunder = "double underscore" (e.g., __len__, __str__)

Python calls them automatically — you never call them directly

They make your objects feel native — users use normal Python syntax

Each dunder maps to a specific built-in operation</h6>

- example 4.1

In [21]:
class DatasetProfile:
    def __init__(self):
        self._columns: list[ColumnProfile] = []
        # The underscore prefix indicates that this attribute is intended to be private
        # private list to store ColumnProfile objects

    def add_column(self, col: ColumnProfile) -> None:
        self._columns.append(col)
        # Adds a ColumnProfile (or subclass) instance to the private list
        # Regular method to add a column to the internal list

    def __len__(self) -> int:
        # Called automatically by: len(profile)
        return len(self._columns)
        # return how many columns are stored in the private list

    def __contains__(self, name: str) -> bool:
        # Checks if a column with the given name exists in the dataset
        # Called automatically by: 'name' in profile
        return any (col.name == name for col in self._columns)
        # Returns True if ANY column has a matching name

    def __iter__(self):
        # Called automatically by: for col in profile
        return iter(self._columns)
        # Returns an iterator over the private list of columns
        # Allows iteration over the DatasetProfile object itself

    def __getitem__(self, name: str) -> ColumnProfile:
        # called automatically by: profile['name']
        for col in self._columns:
            if col.name == name:
                return col
        raise KeyError(f"no column named {name!r}")
        # Raises KeyError if no column with the given name is found

profile = DatasetProfile()
profile.add_column(ColumnProfile("id", "int64", 0, 5000))
profile.add_column(CategoricalColumn("country", "object", 12, 5000, 87))

print(len(profile))
print('naty' in profile)
print('id' in profile)

for col in profile:
    # itrates because of __iter__ method
    print(col.summary())
    # prints each column's summery

print(profile['id']) # fetches by name because of __getitem__
# → returns the CategoricalColumn object named "country"   

2
False
True
id [int64] — 0.0% null
country [object] — 0.2% null | 87 unique values


**+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++**

**Extra Example for each Concepts**

**+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++**


**Inheritance**

- exercise c1.3

In [10]:
class product:
    def __init__(self, id, name, price, stoke):
        self.id = id
        self.name = name
        self.price = price
        self.stoke = stoke
        self.tax_rate = 0.8

    def price_with_tax(self):
        return self.price * (1 + self.tax_rate)

    def is_available(self):
        return self.stoke > 0 

    def description(self):
        return f"{self.name} (ID:{self.id}) - ${self.price:.2f}"           

class digital_product(product):
    def __init__(self, id, name, price, file_size):
        super().__init__(id, name, price, stoke= 9999)
        self.file_size = file_size
        self.tax_rate = 0.0

    def description(self):
        return f"{super().description()}|digital, {self.file_size}mb"

class physical_product(product):
    def __init__(self, id, name, price, stoke, weight, shipping_cost):
        super().__init__(id, name, price, stoke)
        self.weight = weight
        self.shipping_cost = shipping_cost            

    def total_price(self, quantity=1):
        base_price = self.price * quantity
        tax = base_price * self.tax_rate
        shipping = self.shipping_cost * quantity
        return base_price + tax + shipping
    
    def description(self):  # Override
        return f"{super().description()} | {self.weight}kg"

ebook = digital_product(101, "Python Guide", 29.99, 15)
laptop = physical_product(102, "Laptop", 999.99, 10, 2.5, 25.00)

print(ebook.description())  
# Python Guide (ID: 101) - $29.99 | Digital, 15MB

print(laptop.total_price(2))  
# 999.99*2 + tax (8%) + 25*2 shipping = 2159.98

print(ebook.is_available())  # True (inherited)
print(laptop.is_available()) # True (inherited)
                   

Python Guide (ID:101) - $29.99|digital, 15mb
3649.964
True
True



exercise c1.4

In [11]:
class Vehicle:
    def __init__(self, brand, model, year, price):
        self.brand = brand
        self.model = model
        self.year = year
        self.price = price
        self.is_running = False

    def start(self):
        self.is_running = True
        return f"{self.brand} {self.model} started"

    def stop(self):
        self.is_running = False
        return f"{self.brand} {self.model} stopped"

    def get_age(self):
        from datetime import datetime
        return datetime().year - self.year 

    def info(self):
        return f"brand = {self.brand} , model= {self.model} year({self.year}) ${self.price}"

class Car(Vehicle):
    def __init__(self, brand, model, year, price, fuel_type: str, mileage: float):
        super().__init__(brand, model, year, price)
        self.fuel_type = fuel_type
        self.mileage = mileage 

    def info(self):
        return f"{super().info()} | fuel_type={self.fuel_type} | mileage={self.mileage}" 

class ElectricCar(Car):
    def __init__(self,brand, model, year, price,fuel_type, mileage, battery_capacity: (float), range_miles: (int)):
        super().__init__(brand, model, year, price, fuel_type, mileage)
        self.battery_capacity = battery_capacity
        self.range_miles = range_miles

    def start(self):
        return f"{super().start()} | electric cars have more advantage on noise decreasing"  
    
    def charge(self):
        return f"{self.brand} {self.model} is charging..."

class MotorCycle(Vehicle):
    def __init__ (self, brand, model, year, price, engine_cc: int, has_sidecar: bool):
        super().__init__(brand, model, year, price)
        self.engine_cc = engine_cc
        self.has_sidecar = has_sidecar

    def info(self):
        return f"{super().info()} | {self.engine_cc}"    

car = Car('toyota', 'y5', 2002, 400, 'diesel', 2.3)    
toy = ElectricCar('gta', 'cc50', 2020, 4400, 'nafta', 4.4, 500, True)
f_1 = MotorCycle('gta', 'cc2', 2020, 111000, 22, True)

print(car.info())
print(toy.start())
print(f_1.info())
print(toy.charge())
print()



brand = toyota , model= y5 year(2002) $400 | fuel_type=diesel | mileage=2.3
gta cc50 started | electric cars have more advantage on noise decreasing
brand = gta , model= cc2 year(2020) $111000 | 22
gta cc50 is charging...



example c1.5

In [12]:
class LibraryItem:
    """Base class for all library items"""
    def __init__(self, item_id, title, year, available=True):
        self.item_id = item_id
        self.title = title
        self.year = year
        self.available = available
        self.checked_out_by = None

    def checked_out(self, patron_name):
        if not self.available:
            return f"{self.title} is not available."
        self.available = False
        self.checked_out_by = patron_name
        return f"{self.title} is checked out by {patron_name}"

    def return_item(self):
        if self.available:
            return f"{self.title} is already available."
        self.available = True
        patron_name = self.checked_out_by
        self.checked_out_by = None 
        return f"{self.titla} is returned by {patron_name}"

    def get_info(self):
        status = "Available" if self.available else f"Checked out by {self.checked_out_by}"
        return f"{self.title} ({self.year}) - ID: {self.item_id} - {status}"

class Book(LibraryItem):
    def __init__(self, item_id, title, year, available, author, page, isbn):
        super().__init__(item_id, title, year, available)
        self.author = author
        self.page = page
        self.isbn = isbn

    def get_info(self):
        return f"{super().get_info()} - [{self.author} | {self.page}]"        

class DVD(LibraryItem):
    def __init__(self, item_id, title, year, available, director: (str), duration: (int), rating: (str)):
        super().__init__(item_id, title, year, available)
        self.director = director
        self.duration = duration
        self.rating = rating

    def get_info(self):
        return f"{super().get_info()}- [{self.director} | {self.duration} min]" 

    def play(self):
        return f"Playing {self.title} | enjoy the movie !" 

class magazine(LibraryItem):
    def __init__ (self, item_id, title, year, available, issue_number: (int), publisher: (str)):
        super().__init__(item_id, title, year, available)     
        self.issue_number = issue_number
        self.publisher = publisher

    def get_info(self):
        return f"{super().get_info()} - {self.issue_number}"


book = Book("B001", "Python Programming", 2021, True, "John Doe", 350, "978-3-16-148410-0")
dvd = DVD("D001", "Inception", 2010, True, "Christopher Nolan", 148, "PG-13")

print(book.get_info())
print(dvd.get_info())
print(dvd.play())
print(book.checked_out('alice'))

Python Programming (2021) - ID: B001 - Available - [John Doe | 350]
Inception (2010) - ID: D001 - Available- [Christopher Nolan | 148 min]
Playing Inception | enjoy the movie !
Python Programming is checked out by alice


example c1.6

In [13]:
class Employee:
    def __init__ (self, emp_id, name, email, base_salary):
        self.emp_id = emp_id
        self.name = name
        self.email = email
        self.base_salary = base_salary
        self.hours_worked = 0

    def work(self, hours):
        self.hours_worked += hours    
        return f"{self.name} worked {self.hours_worked} hours"

    def calculate_pay(self):
        return self.base_salary

    def get_info(self): 
        return f"ID: {self.emp_id} - {self.name} - ${self.base_salary}"

class FullTimeEmployee(Employee):
    def __init__ (self, emp_id, name, email, base_salary, annual_bonus: (float), benefits: (list)) :
        super().__init__ (emp_id, name, email, base_salary)
        self.annual_bonus = annual_bonus
        self.benefits = benefits       

    def calculate_pay(self):
        #salary_by_month = (self.base_salary + self,annual_bonus) / 12
        return (self.base_salary + self.annual_bonus) / 12

    def get_info(self):
        total = (self.base_salary) + (self.annual_bonus)
        return f"{super().get_info()} + bonus = {self.annual_bonus}$, total={total} | {self.benefits}"    

class PartTimeEmployee(Employee):
    def __init__ (self, emp_id, name, email, base_salary, hourly_rate: (float), max_hours_per_week: (int)):
        super().__init__(emp_id, name, email, base_salary) 
        self.hourly_rate = hourly_rate       
        self.max_hours_per_week = max_hours_per_week

    def calculate_pay(self):
        self.base_salary = self.hourly_rate * self.hours_worked

    def work(self, hours):
        if self.hours_worked + hours > self.max_hours_per_week * 4:  # Monthly check
            return f"Cannot work {hours} hours. Exceeds monthly limit"
        return super().work(hours)

class Manager(FullTimeEmployee):
    def __init__ (self, emp_id, name, email, base_salary, annual_bonus, benefits, team_size: (int), department: (str)):
        super().__init__ (emp_id, name, email, base_salary, annual_bonus, benefits)
        self.team_size = team_size
        self.department = department

    def hire_employee(self):
        self.team_size += 1
        self.team_members.append(employee_name)
        return f"Congrats ! {self.employee_name}, your are hired to the company."    

    def get_team_report(self):
        members = ", ".join(self.team_members) if self.team_members else "No team members yet"
        return f"{self.department} team: {self.team_size} members - {members}"   

fulltime = FullTimeEmployee(101, "Alice", "alice@company.com", 60000, 5000, ["Health", "Dental"])

print(fulltime.calculate_pay())

5416.666666666667


**===============================================================**

**polymorphism — one interface, many behaviors**

- exercise c2.2

In [14]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age
        self.hunger = 50

    def make_sound(self):
        return "something generic"

    def eat(self, food_amount):
        self.hunger = max(0, self.hunger - food_amount)
        return f"{self.name} ate {food_amount} food, hunger: {self.hunger}"

    def move(self):
        """How the animal moves - to be overridden"""
        return f"{self.name} moves in some way"
    
    def info(self):
        return f"{self.name} ({self.age} years old), hunger: {self.hunger}"
    
class Dog(Animal):
    def __init__(self, name, age):
        super().__init__(name, age)

    def make_sound(self):
        return f"{self.name} says woof!"

    def move(self):
        return f"{self.name} runs on four legs"

    def fetch(self, item):
        return f"{self.name} fetches the {item}"

class Cat(Animal):
    def __init__(self, name, age):
        super().__init__(name, age)

    def make_sound(self):
        return f"{self.name} says meow!"

    def move(self):
        return f"{self.name} walk gracefully"

    def purr(self):
        return f"{self.name} is purring"                       

class Bird(Animal):
    def __init__(self, name, age):
        super().__init__(name, age)

    def make_sound(self):
        return "Chirp!"

    def move(self):
        return f"{self.name} flies through the air"

    def fly(self, distance):
        return f"{self.name} flew {distance} meters" 

class Fish(Animal):
    def __init__(self, name, age):
        super().__init__(name, age)

    def make_sound(self):
        return "Blub!"

    def move(self):
        return f"{self.name} swims in the water"

    def swim(self, depth):
        return f"{self.name} swam {depth} meters"

animals = [
    Dog('bolt', 3),
    Cat('garfiled', 4),
    Bird('parrot', 2),
    Fish('Nemo', 6)
] 

for a in animals:
    print(a.make_sound())
    print(a.move())
    print(a.info())
    print('-'*40) 

print('*'*40)    

def feed_animal(animal, amount):
    print(animal.eat(amount))

feed = [
    (Dog('rex', 5), 20),
    (Cat('whiskers', 3), 15),
    (Bird('tweety', 1), 10),
    (Fish('goldie', 2), 5)
]    

for animal, amount in feed:
    feed_animal(animal, amount)


    

bolt says woof!
bolt runs on four legs
bolt (3 years old), hunger: 50
----------------------------------------
garfiled says meow!
garfiled walk gracefully
garfiled (4 years old), hunger: 50
----------------------------------------
Chirp!
parrot flies through the air
parrot (2 years old), hunger: 50
----------------------------------------
Blub!
Nemo swims in the water
Nemo (6 years old), hunger: 50
----------------------------------------
****************************************
rex ate 20 food, hunger: 30
whiskers ate 15 food, hunger: 35
tweety ate 10 food, hunger: 40
goldie ate 5 food, hunger: 45


exercise c2.3

In [15]:
import math

class Shape:
    def __init__(self, color='black'):
        self.color = color

    def area(self):
        raise NotImplementedError("Subclasses must implement area()")

    def perimeter(self):
        raise NotImplementedError("Subclasses must implement perimeter()")

    def describe(self):
        return f"{self.color} {self.__class__.__name__.lower()}"

class Rectangle(Shape):
    def __init__(self, width, height, color='black'):
        super().__init__(color)
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, color, radius):
        super().__init__(color)
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius

class Triangle(Shape):
    def __init__(self, color, side_a, side_b, side_c):
        super().__init__(color)
        self.side_a = side_a
        self.side_b = side_b
        self.side_c = side_c

    def area(self):
        s = (self.side_a + self.side_b + self.side_c) / 2
        return math.sqrt(s * (s - self.side_a) * (s - self.side_b) * (s - self.side_c))

    def perimeter(self):
        return self.side_a + self.side_b + self.side_c                

class Square(Rectangle):
    def __init__(self, color, side_length):
        super().__init__(side_length, side_length, color)

shapes = [
    Rectangle(3, 4, 'red'),
    Circle('blue', 5),
    Triangle('green', 3, 4, 5),
    Square('yellow', 2)
]        
        
def total_area(shapes):
    return sum(shape.area() for shape in shapes)

def largest_perimeter(shapes):
    return max(shapes, key=lambda shape: shape.perimeter())

print("Shape details:")
for shape in shapes:
    print(f"- {shape.describe()}: Area = {shape.area():.2f}, Perimeter = {shape.perimeter():.2f}")
print('*'*40)
print(f"Total area: {total_area(shapes):.2f}")
print(f"Largest perimeter: {largest_perimeter(shapes).describe()} with perimeter = {largest_perimeter(shapes).perimeter():.2f}")

Shape details:
- red rectangle: Area = 12.00, Perimeter = 14.00
- blue circle: Area = 78.54, Perimeter = 31.42
- green triangle: Area = 6.00, Perimeter = 12.00
- yellow square: Area = 4.00, Perimeter = 8.00
****************************************
Total area: 100.54
Largest perimeter: blue circle with perimeter = 31.42


**===============================================================**

**isinstance and issubclass**

- exercise c3.3

In [9]:
import math

class Shape:
    """Base shape class"""
    def __init__(self, color="white"):
        self.color = color
    
    def area(self):
        raise NotImplementedError("Subclasses must implement area()")
    
    def perimeter(self):
        raise NotImplementedError("Subclasses must implement perimeter()")
    
    def __str__(self):
        return f"{self.color} {self.__class__.__name__.lower()}"

class Circle(Shape):
    def __init__(self, color, radius):
        super().__init__(color)
        self.radius = radius
    
    def area(self):
        return math.pi * self.radius ** 2
    
    def perimeter(self):
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, color, width, height):
        super().__init__(color)
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height
    
    def perimeter(self):
        return 2 * (self.width + self.height)

class Square(Rectangle):
    def __init__(self, color, side):
        super().__init__(color, side, side)

class Triangle(Shape):
    def __init__(self, color, side1, side2, side3):
        super().__init__(color)
        self.side1 = side1
        self.side2 = side2
        self.side3 = side3
    
    def area(self):
        s = self.perimeter() / 2
        return math.sqrt(s * (s - self.side1) * (s - self.side2) * (s - self.side3))
    
    def perimeter(self):
        return self.side1 + self.side2 + self.side3

# TODO: Create a function that prints shape information with type checking
def print_shape_info(shape):
    """Print detailed information about a shape"""
    print(f"📐 Shape: {shape}")
    print("-" * 40)
    
    # TODO: Use isinstance to identify the shape type
    # If it's a Circle: print "   Type: Circle with radius X"
    if isinstance(shape, Circle):
        print(f"   Type: Circle with radius {shape.radius}")
    # If it's a Rectangle: print "   Type: Rectangle X x Y"
    elif isinstance(shape, Rectangle):
        print(f"   Type: Rectangle {shape.width} x {shape.height}") 
    # If it's a Square: print "   Type: Square with side X"
    elif isinstance(shape, Square):
        print(f"   Type: Square with side {shape.width}")
    # If it's a Triangle: print "   Type: Triangle with sides X, Y, Z"
    elif isinstance(shape, Triangle):
        print(f"   Type: Triangle with sides {shape.side1}, {shape.side2}, {shape.side3}")

    # Common information for all shapes
    print(f"   Area: {shape.area():.2f}")
    print(f"   Perimeter: {shape.perimeter():.2f}")
    print(f"   Color: {shape.color}")

# TODO: Create a function that processes multiple shapes
def process_shapes(shapes):
    """Process a list of shapes with special handling"""
    for shape in shapes:
        print_shape_info(shape)
        print("-" * 40)

# TODO: Test with different shapes
shapes = [
    Circle("red", 5),
    Rectangle("blue", 4, 6),
    Square("green", 3),
    Triangle("yellow", 3, 4, 5)
]

process_shapes(shapes)

# TODO: Challenge: Write a function that:
# 1. Finds all circles in a list of shapes
def find_circles(shapes):
    if isinstance(shapes, list):
        for shape in shapes:
            if isinstance(shape, Circle):
                return shape
    return f'shape is not in cicles'

# 2. Finds all rectangles (including squares)
def find_rectangle(shapes):
    if isinstance(shapes, list):
        for shape in shapes:
            if isinstance(shape, Rectangle):
                if isinstance(shape, Square):
                    return shape
    return f'shape is not in rectangles'
    
# 3. Calculates total area of only circles
def calculate_total_circle_area(shapes):
    total_area = 0
    for shape in shapes:
        if isinstance(shape, Circle):
            total_area += shape.area()
    return total_area

print(find_circles(shapes)) 
print(find_rectangle(shapes)) 
print(calculate_total_circle_area(shapes))


📐 Shape: red circle
----------------------------------------
   Type: Circle with radius 5
   Area: 78.54
   Perimeter: 31.42
   Color: red
----------------------------------------
📐 Shape: blue rectangle
----------------------------------------
   Type: Rectangle 4 x 6
   Area: 24.00
   Perimeter: 20.00
   Color: blue
----------------------------------------
📐 Shape: green square
----------------------------------------
   Type: Rectangle 3 x 3
   Area: 9.00
   Perimeter: 12.00
   Color: green
----------------------------------------
📐 Shape: yellow triangle
----------------------------------------
   Type: Triangle with sides 3, 4, 5
   Area: 6.00
   Perimeter: 12.00
   Color: yellow
----------------------------------------
red circle
green square
78.53981633974483


**===============================================================**

**Dunder Methods**

- exercise c4.2

In [23]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def __str__(self):
        return f"{self.title} by {self.author}"

class Bookshelf:
    def __init__(self):
        self._books = []

    def add(self, book):
        self._books.append(book)

    def __len__(self) -> int:
        return len(self._books)

    def __contains__(self, title:str) -> bool:
        return title in self._books 

    def __iter__(self):
        return iter(self._books)

    def __getitem__(self, title:str) -> Book:
        return self._books[title]
        
shelf = Bookshelf()
shelf.add(Book("Python 101", "Alice"))
shelf.add(Book("OOP Guide", "Bob"))
shelf.add(Book("Data Structures", "Carol"))

print(len(shelf))                  # Expected: 3
print("Python 101" in shelf)       # Expected: True
print("Java 101" in shelf)         # Expected: False

for book in shelf:
    print(book)                    # Expected: prints all 3 books

print(shelf[1])                    # Expected: OOP Guide by Bob                                       

3
False
False
Python 101 by Alice
OOP Guide by Bob
Data Structures by Carol
OOP Guide by Bob


In [7]:
class Cart:
    def __init__(self):
        self._items = []

    def add(self, item):
        self._items.append(item)

    def __len__(self):
        return len(self._items)

    def __contains__(self, item):
        return item in self._items 

    def __iter__(self):
        return iter(self._items)

    def __getitem__(self, index):
        return self._items[index]

cart = Cart()
cart.add('apple')
cart.add('bread')
cart.add('milk')

print(len(cart))
print('bread' in cart)
print('egg' in cart)

for item in cart:
    print(item)


print(cart[0])
print(cart[-1])



3
True
False
apple
bread
milk
apple
milk


In [24]:
class Task:
    def __init__(self, title):
        self.title = title

    def __str__(self):
        return self.title

class TodoList:
    def __init__(self):
        self._tasks = []

    def add(self, task):
        self._tasks.append(task)

    def __len__(self):
        return len(self._tasks)

    def __contains__(self, title):
        return any(task.title == title for task in self._tasks)

    def __iter__(self):
        return iter(self._tasks)

    def __getitem__(self, index) -> TodoList:
        return self._tasks[index]

todos = TodoList()
todos.add(Task('buy grocery'))
todos.add(Task('walk the dog'))
todos.add(Task('work out'))
todos.add('drink water')

print(len(todos))
print('drink' in todos)
print('work out' in todos)

for task in todos:
    print(task)

print(todos[3])    



4
False
True
buy grocery
walk the dog
work out
drink water
drink water


In [28]:
class Contact:
    def __init__(self, name, phone):
        self.name = name
        self.phone = phone

    def __str__(self):
        return f"{self.name}: {self.phone}"

class Contactbook:
    def __init__(self):
        self._contacts = []

    def add(self, contact):
        self._contacts.append(contact)

    def __len__(self):
        return len(self._contacts)

    def __contains__(self, name: str) -> bool:
        return any(contact.name == name for contact in self._contacts)

    def __iter__(self):
        return iter(self._contacts)

    def __getitem__(self, index):
        return self._contacts[index]

phone =  Contactbook()
phone.add(Contact('naty', '0909'))
phone.add(Contact('bob', '1111'))
phone.add(Contact('dada', '2222'))

print(len(phone))
print('naty' in phone)
print('aster' in phone)
print('0909' in phone)

for i in phone:
    print (i)
print('*' * 10)
print (phone[0])


3
True
False
False
naty: 0909
bob: 1111
dada: 2222
**********
naty: 0909
